# Deploying AI-Powered Research Servers


In this project, I implemented a remote server using `FastMCP` to handle research queries. The server leverages the `arxiv` library to fetch academic papers based on user-defined topics. The project also demonstrates the use of `sse` transport for efficient communication. This implementation highlights my understanding of remote server deployment and AI integration.

## Creating an SSE Remote Server

With `FastMCP`, it's also easy to create an SSE remote server. You just need to specify that the transport is `sse` when running the server. You can also specify the port number when initializing the FastMCP server. The remaining tool, prompt and resource definitions are all the same. So the following code for the MCP server is the same code you saw in Lesson 7. The transport is specified at the end, and the port number of `8001` is specified in the `FastMCP` constructor (you may also decide to choose the default port as well). 

In [ ]:
# Write the research server implementation to a file
import arxiv
import json
import os
from typing import List
from mcp.server.fastmcp import FastMCP

PAPER_DIR = "papers"

# Initialize FastMCP server
mcp = FastMCP("research", port=8001)

@mcp.tool()
def search_papers(topic: str, max_results: int = 5, year_filter: int = None) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    Optionally filter papers by publication year.

    Args:
        topic: The topic to search for.
        max_results: Maximum number of results to retrieve (default: 5).
        year_filter: Filter papers published in or after this year (default: None).

    Returns:
        List of paper IDs found in the search.
    """

    # Use arxiv to find the papers 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query=topic,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)

    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)

    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    # Process each paper and add to papers_info
    filtered_papers = []
    for paper in papers:
        if year_filter and paper.published.year < year_filter:
            continue

        paper_id = paper.entry_id
        papers_info[paper_id] = {
            "title": paper.title,
            "authors": [author.name for author in paper.authors],
            "summary": paper.summary,
            "published": paper.published.strftime("%Y-%m-%d"),
            "url": paper.entry_id
        }
        filtered_papers.append(paper_id)

    # Save updated papers info
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=4)

    return filtered_papers

This project demonstrates how to create and deploy a remote research server using `FastMCP`. The server is designed to search for academic papers on arXiv and store their information. This implementation highlights my ability to build scalable and modular AI-powered systems.

## Testing the SSE Remote Server

After you create the python file for your remote server, you can test it using the MCP inspector or the simple chatbot of lesson 5, you can also integrate it with the chatbot of lesson 7. In order to test it, you first need to launch it to get its `URL` and then provide the `URL` to the chatbot or MCP inspector. 

**Note**: A server using the `stdio` transport is launched as a subprocess by the MCP client. On the other hand, a remote server is an independent processes running separately from the client and needs to be already running before the MCP client connects to it.


### How to run and test it on your local machine? - Optional Reading

You would need first to create and prepare a separate environment for the remote server. You can follow the same steps you learned about in the previous lessons:
- initiate the folder using `uv init`,
- create a virtual environment and activate it,
- add the required dependencies (`uv add arxiv mcp`).

You can then open a terminal and run your server using (`uv run research_server.py`), you'd need to keep the terminal open for the server to keep on running. You'll get a message in the terminal that the server is running at a given address. The `URL` that you would need to provide to the inspector or chatbot is that address with the appended `/sse` at the end. In a second terminal, you can launch the MCP inspector or your chatbot. Please check the comments below on how you can update the MCP chatbot. 

**Streamable HTPP**: same process for everything. For the `URL`, you would need to append `/mcp/` instead of `/sse`.

### Testing the server with MCP Inspector in this Lab

#### URL Link to the Remote server

In this lesson, you won't need to launch the remote server on your own. It's already provided to you; the remote server is already up and running in a separate container at port 8001. Run the next cell to get its URL. You'd need to use this URL to test the server in the MCP inspector. 

**Note**: If you'd like to learn how to run the MCP server in a docker container, please check the Appendix at the end of this course.

In [ ]:
import os 
print("Remote server is running at:")
print(os.environ.get('DLAI_LOCAL_URL').format(port=8001)+"sse")

#### Testing the Provided Remote Server using MCP Inspector

**Terminal Instructions**

- To open the terminal, run the cell below.
  - The terminal might show the `work` directory or `L4/mcp_project`. You can stay in any directory, you don't need to navigate to `L9/mcp_project`. That's because you'll test a remote server that is already up and running, so you can launch the inspector from any directory.
  - If the terminal shows `L4/mcp_project` and you still have the inspector open from L4, you can close it by typing `CTRL+C` and then launch it again.
- To launch the inspector, type in the terminal: `npx @modelcontextprotocol/inspector`
  - If you get a message asking "need to install the following packages", type: `y`
- You will get a message saying that the inspector is up and running at a specific address. To open the inspector, click on that given address. The inspector will open in another tab.
- Please check the "Inspector UI Instructions" below.
- Once you're done with the inspector UI, make sure to close the inspector by typing `Ctrl+C` in the terminal below.

**Note**: The server is running in a different environment, and there are no topics saved under papers in the server's environment, so when you check the resources in the MCP inspector, you will get that there are no resources. 

In [ ]:
# start a new terminal
from IPython.display import IFrame

IFrame(f"{os.environ.get('DLAI_LOCAL_URL').format(port=8888)}terminals/1", width=600, height=768)

**Inspector UI Instructions**

In the inspector UI, make sure you have:

<img src="images/inspector2.png" height="300">

1.  `SSE` under `Transport Type`
2.  The URL of the remote server under `URL` (this is the link that ends with `sse`, the output of the cell you run before the terminal)
3.  Under configuration, you have to specify the "Inspector Proxy Address":
      - Run the following cell and copy the output address and paste it under "Inspector Proxy Address" in the inspector UI. Note: if you're running the inspector locally on your machine, you don't need to worry about this step.

In [ ]:
# Print the inspector proxy address
print("Inspector Proxy Address that you need to specify under configuration in the inspector UI:")
print(os.environ.get('DLAI_LOCAL_URL').format(port=6277)[:-1])

#### Optional Note - How to Update the MCP chatbot so it can connect to a remote server?

## Optional - Deploying an SSE Server

This section explores how to deploy an SSE server locally. Follow the instructions to set up the environment and test the deployment process.

## Optional - Deploying an SSE Server Using Render.com 

This part is optional and for you to explore on your local machine. If you'd like to try it, you can follow the instructions in the video. Make sure first to create an account at render [here](https://dashboard.render.com/login) using your Github account. 

## Resources

- Deploy Remote MCP servers on CloudFlare [link](https://developers.cloudflare.com/agents/guides/remote-mcp-server/)
- Streamable HTTP transport [link](https://github.com/modelcontextprotocol/python-sdk/blob/main/README.md#streamable-http-transport)
- For low level server with Streamable HTTP implementations, see:
    - Stateful server: [examples](https://github.com/modelcontextprotocol/python-sdk/tree/main/examples/servers/simple-streamablehttp)
    - Stateless server: [examples](https://github.com/modelcontextprotocol/python-sdk/tree/main/examples/servers/simple-streamablehttp-stateless)